In [1]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Aave V3.1 — feature addition

Loads the **transformed** frames from `transformed_data/`:

- `DF_common_1` — asset-level reserve rates/indexes, keyed on `(time_bucket, asset)`
- `DF_common_final` — protocol-level 6h time series, keyed on `time_bucket`

In [2]:
# Load transformed data from transformed_data/
import pandas as pd
from pathlib import Path
from IPython.display import display
import adv_validation as adv

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 10

DF_common_1 = pd.read_csv(DATA_DIR / "DF_common_1.csv")          # asset-level (time_bucket, asset)
DF_common_final = pd.read_csv(DATA_DIR / "DF_common_final.csv")  # protocol-level 2h series

# separate sparse-family panels (2h as-is + 24h / 7d rollups) — kept apart from DF_common_*
df_liq_2h  = pd.read_csv(DATA_DIR / "df_liq_2h.csv")
df_liq_24h = pd.read_csv(DATA_DIR / "df_liq_24h.csv")
df_liq_7d  = pd.read_csv(DATA_DIR / "df_liq_7d.csv")
df_user_account_2h  = pd.read_csv(DATA_DIR / "df_user_account_2h.csv")
df_user_account_24h = pd.read_csv(DATA_DIR / "df_user_account_24h.csv")
df_user_account_7d  = pd.read_csv(DATA_DIR / "df_user_account_7d.csv")

print(f" {DF_common_final.shape[0]} rows x {DF_common_final.shape[1]} cols")
display(DF_common_final.head(PREVIEW_ROWS))

# ── input guards (ERR-15): refuse a stale / unrepaired / untrimmed transform export ──
_event_cols = [c for c in DF_common_final.columns if c != "time_bucket"]
assert DF_common_final[_event_cols].notna().all().all(), \
    "unrepaired export: nulls in event columns — re-run transform.ipynb export cell"
assert DF_common_final["time_bucket"].max() < "2026-03-31", \
    "untrimmed export: unpriced last-day buckets present (ERR-03) — re-run transform.ipynb"
assert "user_state_observed" in df_user_account_24h.columns, \
    "user-state audit metadata missing (ERR-17) — re-run transform.ipynb"
print(" input guards OK: repaired, trimmed, user-state metadata present")


 4368 rows x 30 cols


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,unique_collateral_enable_users,unique_collateral_disable_users,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-04-01 00:00:00.000 UTC,107,112,54,60,1.814934e+08,99340.849932,9.766776e+07,53458.667287,54.0,...,59,56,9.547419e+05,522.580146,248.230641,0.135870,26.0,20.0,19.0,7.0
1,2025-04-01 02:00:00.000 UTC,116,105,60,55,2.512122e+08,136862.261136,2.665668e+08,145227.537741,58.0,...,61,41,6.532868e+06,3559.148456,3038.132814,1.655194,23.0,19.0,19.0,4.0
2,2025-04-01 04:00:00.000 UTC,72,100,38,54,1.096440e+08,59557.933326,1.329771e+08,72232.382778,48.0,...,41,36,2.605000e+05,141.502472,8.453992,0.004592,9.0,7.0,8.0,1.0
3,2025-04-01 06:00:00.000 UTC,129,115,58,48,6.726555e+08,362808.719631,6.691233e+08,360903.546607,64.0,...,58,39,1.136226e+06,612.843849,147.875707,0.079760,17.0,7.0,6.0,11.0
4,2025-04-01 08:00:00.000 UTC,138,118,68,53,3.680422e+08,195833.482106,3.695117e+08,196615.376327,95.0,...,68,43,2.134066e+06,1135.526324,708.085451,0.376769,28.0,17.0,17.0,11.0
5,2025-04-01 10:00:00.000 UTC,127,111,77,59,2.532615e+08,135086.519080,2.552678e+08,136156.675648,72.0,...,62,57,2.762158e+06,1473.313495,235.541791,0.125635,15.0,12.0,11.0,4.0
6,2025-04-01 12:00:00.000 UTC,127,122,60,62,4.748889e+08,254080.767024,4.601955e+08,246219.340371,57.0,...,63,58,9.532554e+05,510.022369,194.910465,0.104283,17.0,12.0,14.0,3.0
7,2025-04-01 14:00:00.000 UTC,142,111,66,60,5.869603e+08,311246.523260,4.781272e+08,253535.842057,98.0,...,61,51,1.710213e+06,906.836451,400.869811,0.212556,36.0,21.0,17.0,19.0
8,2025-04-01 16:00:00.000 UTC,122,103,62,55,1.321209e+08,68913.296488,1.362937e+08,71089.779751,82.0,...,49,44,4.495563e+05,234.485268,68.061213,0.035500,24.0,13.0,11.0,13.0
9,2025-04-01 18:00:00.000 UTC,137,98,54,55,5.251323e+08,275222.083697,5.202348e+08,272655.318074,97.0,...,46,35,4.160916e+05,218.073803,89.506585,0.046910,30.0,16.0,25.0,5.0


 input guards OK: repaired, trimmed, user-state metadata present


In [3]:
# DF_common_final.dtypes.rename("dtype").to_frame()
cols = DF_common_final.columns

temp_stats = adv.statistical_validation(DF_common_final, columns = cols, save = False)

display(temp_stats[["column", "null_pct", "zero_pct", "negative_pct",]])

,column,null_pct,zero_pct,negative_pct
0,supply_tx_count,0.0,0.0000,0.0
1,withdrawal_tx_count,0.0,0.0000,0.0
2,unique_suppliers,0.0,0.0000,0.0
3,unique_withdraw_users,0.0,0.0000,0.0
4,supply_amount_value_usd,0.0,0.0000,0.0
5,supply_amount_value_eth,0.0,0.0000,0.0
6,withdrawal_amount_value_usd,0.0,0.0000,0.0
7,withdrawal_amount_value_eth,0.0,0.0000,0.0
8,borrow_tx_count,0.0,0.0000,0.0
9,repay_tx_count,0.0,0.0000,0.0


In [4]:
# additional liquidity metrics

DF_common_final["net_liquidity_flow_usd"] = DF_common_final["supply_amount_value_usd"] - DF_common_final["withdrawal_amount_value_usd"]
DF_common_final["net_liquidity_flow_eth"] = DF_common_final["supply_amount_value_eth"] - DF_common_final["withdrawal_amount_value_eth"]

# Supply/Withdrawal Ratio
DF_common_final["supply_withdrawal_ratio"] = DF_common_final["supply_amount_value_usd"] / DF_common_final["withdrawal_amount_value_usd"]
# DF_common_final["supply_withdrawal_ratio_eth"] = DF_common_final["supply_amount_value_eth"] / DF_common_final["withdrawal_amount_value_eth"]

# Liquidity Growth Rate
DF_common_final["liquidity_growth_rate"] = (
    DF_common_final["net_liquidity_flow_usd"] - DF_common_final["net_liquidity_flow_usd"].shift(1)
) / DF_common_final["net_liquidity_flow_usd"].shift(1).abs()

# Average Supply Size
DF_common_final["avg_supply_size_usd"] = DF_common_final["supply_amount_value_usd"] / DF_common_final["supply_tx_count"]

# Average Withdrawal Size
DF_common_final["avg_withdrawal_size_usd"] = DF_common_final["withdrawal_amount_value_usd"] / DF_common_final["withdrawal_tx_count"]

new_cols = [
    "net_liquidity_flow_usd",
    "net_liquidity_flow_eth",
    "supply_withdrawal_ratio",
    # "supply_withdrawal_ratio_eth",
    "liquidity_growth_rate",
    "avg_supply_size_usd",
    "avg_withdrawal_size_usd",
]

DF_common_final[["time_bucket"] + new_cols]

temp = adv.statistical_validation(DF_common_final, columns = new_cols, save=False)
display(temp[["column", "null_pct", "zero_pct", "negative_pct",]])

,column,null_pct,zero_pct,negative_pct
0,net_liquidity_flow_usd,0.0000,0.0,46.3828
1,net_liquidity_flow_eth,0.0000,0.0,46.3828
2,supply_withdrawal_ratio,0.0000,0.0,0.0000
3,liquidity_growth_rate,0.0229,0.0,51.7289
4,avg_supply_size_usd,0.0000,0.0,0.0000
5,avg_withdrawal_size_usd,0.0000,0.0,0.0000


In [5]:
# additional borrow metrics
#avg_collateral_base col has null values, so adjusted for it
DF_common_final["net_borrow_demand_usd"] = DF_common_final["borrow_amount_value_usd"] - DF_common_final["repay_amount_value_usd"]
DF_common_final["net_borrow_demand_eth"] = DF_common_final["borrow_amount_value_eth"] - DF_common_final["repay_amount_value_eth"]

DF_common_final["borrow_repay_ratio"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["repay_amount_value_usd"].replace(0, pd.NA)

DF_common_final["avg_borrow_size_usd"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)
DF_common_final["avg_borrow_size_eth"] = DF_common_final["borrow_amount_value_eth"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)

DF_common_final["avg_repay_size_usd"] = DF_common_final["repay_amount_value_usd"] / DF_common_final["repay_tx_count"].replace(0, pd.NA)
DF_common_final["avg_repay_size_eth"] = DF_common_final["repay_amount_value_eth"] / DF_common_final["repay_tx_count"].replace(0, pd.NA)

DF_common_final["borrow_growth"] = (
    DF_common_final["borrow_amount_value_usd"] - DF_common_final["borrow_amount_value_usd"].shift(1)
) / DF_common_final["borrow_amount_value_usd"].shift(1).replace(0, pd.NA)

# DF_common_final["debt_expansion_ratio"] = DF_common_final["avg_total_debt_base"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)

borrow_cols = [
    "net_borrow_demand_usd",
    "net_borrow_demand_eth",
    "borrow_repay_ratio",
    "avg_borrow_size_usd",
    "avg_borrow_size_eth",
    "avg_repay_size_usd",
    "avg_repay_size_eth",
    "borrow_growth",
    # "debt_expansion_ratio",
]

DF_common_final[["time_bucket"] + borrow_cols]

,time_bucket,net_borrow_demand_usd,net_borrow_demand_eth,borrow_repay_ratio,avg_borrow_size_usd,avg_borrow_size_eth,avg_repay_size_usd,avg_repay_size_eth,borrow_growth
0,2025-04-01 00:00:00.000 UTC,7.778764e+23,4.257724e+20,1.682295,3.551785e+22,1.944077e+19,2.425718e+22,1.327722e+19,NaN
1,2025-04-01 02:00:00.000 UTC,1.840865e+24,1.002915e+21,1.470134,9.924965e+22,5.407182e+19,6.751060e+22,3.678019e+19,2.001349
2,2025-04-01 04:00:00.000 UTC,-7.023284e+24,-3.815013e+21,0.227876,4.318279e+22,2.345670e+19,2.021346e+23,1.097985e+20,-0.639923
3,2025-04-01 06:00:00.000 UTC,-1.864056e+25,-1.005409e+22,0.068227,2.132690e+22,1.150305e+19,2.985894e+23,1.610490e+20,-0.341500
4,2025-04-01 08:00:00.000 UTC,3.902943e+24,2.076744e+21,2.032262,8.088321e+22,4.303765e+19,5.325299e+22,2.833567e+19,4.629557
...,...,...,...,...,...,...,...,...,...
4363,2026-03-30 14:00:00.000 UTC,2.124313e+25,1.029515e+22,28.052138,1.966821e+23,9.531890e+19,7.774911e+21,3.768007e+18,4.949219
4364,2026-03-30 16:00:00.000 UTC,-6.462694e+24,-3.140216e+21,0.256831,1.976490e+22,9.603799e+18,8.965081e+22,4.356129e+19,-0.898611
4365,2026-03-30 18:00:00.000 UTC,1.805539e+24,8.900852e+20,1.666076,4.342548e+22,2.140752e+19,2.946424e+22,1.452496e+19,1.022110
4366,2026-03-30 20:00:00.000 UTC,3.019209e+24,1.487743e+21,1.116860,3.847382e+23,1.895834e+20,3.229520e+23,1.591377e+20,5.389232


In [6]:
# risk_cols_pre = [
#     "avg_total_collateral_base",
#     "avg_total_debt_base",
#     "avg_available_borrows_base",
#     "avg_current_liquidation_threshold",
#     "avg_ltv",
# ]

# temp_pre_1 = adv.statistical_validation(DF_common_final, columns = risk_cols_pre, save=False)
# display(temp_pre_1[["column", "null_pct", "zero_pct", "negative_pct",]])

In [7]:
# #null value columns are used so expecting approximately same pct of null values
# # implemented correction for NULL and zero values so there is no division by zero or such things

# DF_common_final["collateralization_ratio"] = DF_common_final["avg_total_collateral_base"] / DF_common_final["avg_total_debt_base"].replace(0, pd.NA)

# DF_common_final["borrow_capacity_utilization"] = DF_common_final["avg_total_debt_base"] / (
#     DF_common_final["avg_total_debt_base"] + DF_common_final["avg_available_borrows_base"]
# ).replace(0, pd.NA)

# DF_common_final["remaining_borrow_capacity"] = DF_common_final["avg_available_borrows_base"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)

# DF_common_final["risk_buffer"] = DF_common_final["avg_current_liquidation_threshold"] - DF_common_final["avg_ltv"]

# DF_common_final["ltv_utilization"] = DF_common_final["avg_ltv"] / DF_common_final["avg_current_liquidation_threshold"].replace(0, pd.NA)

# DF_common_final["distance_to_liquidation"] = 1 - DF_common_final["ltv_utilization"]

# risk_cols = [
#     "collateralization_ratio",
#     "borrow_capacity_utilization",
#     "remaining_borrow_capacity",
#     "risk_buffer",
#     "ltv_utilization",
#     "distance_to_liquidation",
# ]

# DF_common_final[["time_bucket"] + risk_cols]

In [8]:
# temp_2 = adv.statistical_validation(DF_common_final, columns = risk_cols, save=False)
# display(temp_2[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# # values and their analytics to be dealt later

In [9]:
# #liquidation metrics
# # the cols use the highest null val cols 
# DF_common_final["liquidation_rate"] = DF_common_final["liquidation_tx_count"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)

# # DF_common_final["liquidation_volume_ratio_usd"] = DF_common_final["liquidation_debt_covered_value_usd"] / DF_common_final["borrow_amount_value_usd"].replace(0, pd.NA)
# DF_common_final["liquidation_volume_ratio_eth"] = DF_common_final["liquidation_debt_covered_value_eth"] / DF_common_final["borrow_amount_value_eth"].replace(0, pd.NA)

# DF_common_final["liquidation_severity_usd"] = DF_common_final["liquidated_collateral_value_usd"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)
# DF_common_final["liquidation_severity_eth"] = DF_common_final["liquidated_collateral_value_eth"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

# DF_common_final["avg_liquidation_debt_usd"] = DF_common_final["liquidation_debt_covered_value_usd"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)
# DF_common_final["avg_liquidation_debt_eth"] = DF_common_final["liquidation_debt_covered_value_eth"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

# DF_common_final["liquidator_concentration"] = DF_common_final["unique_liquidators"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

# DF_common_final["liquidation_user_ratio"] = DF_common_final["unique_liquidated_users"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

# liquidation_cols = [
#     "liquidation_rate",
#     # "liquidation_volume_ratio_usd" 
#     "liquidation_volume_ratio_eth",
#     "liquidation_severity_usd", "liquidation_severity_eth",
#     "avg_liquidation_debt_usd", "avg_liquidation_debt_eth",
#     "liquidator_concentration",
#     "liquidation_user_ratio",
# ]

# DF_common_final[["time_bucket"] + liquidation_cols]

In [10]:
# temp_3 = adv.statistical_validation(DF_common_final, columns = liquidation_cols, save=False)
# display(temp_3[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# # values and their analytics to be dealt later

In [11]:
#user metrics, though sampled user are less. So not that effective as other values.

DF_common_final["borrower_participation_rate"] = DF_common_final["unique_borrowers"] / (
    DF_common_final["unique_borrowers"] + DF_common_final["unique_suppliers"]
).replace(0, pd.NA)

DF_common_final["repayment_discipline"] = DF_common_final["unique_repayers"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

DF_common_final["supplier_activity"] = DF_common_final["supply_tx_count"] / DF_common_final["unique_suppliers"].replace(0, pd.NA)

DF_common_final["borrower_activity"] = DF_common_final["borrow_tx_count"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

DF_common_final["collateral_usage_rate"] = DF_common_final["collateral_enabled_count"] / (
    DF_common_final["collateral_enabled_count"] + DF_common_final["collateral_disabled_count"]
).replace(0, pd.NA)

DF_common_final["collateral_adoption_rate"] = DF_common_final["unique_collateral_enable_users"] / DF_common_final["unique_suppliers"].replace(0, pd.NA)

user_cols= [
    "borrower_participation_rate",
    "repayment_discipline",
    "supplier_activity",
    "borrower_activity",
    "collateral_usage_rate",
    "collateral_adoption_rate",
]

DF_common_final[["time_bucket"] + user_cols]

,time_bucket,borrower_participation_rate,repayment_discipline,supplier_activity,borrower_activity,collateral_usage_rate,collateral_adoption_rate
0,2025-04-01 00:00:00.000 UTC,0.406593,0.729730,1.981481,1.459459,0.508380,1.092593
1,2025-04-01 02:00:00.000 UTC,0.439252,0.893617,1.933333,1.234043,0.573333,1.016667
2,2025-04-01 04:00:00.000 UTC,0.493333,1.027027,1.894737,1.297297,0.524194,1.078947
3,2025-04-01 06:00:00.000 UTC,0.425743,0.837209,2.224138,1.488372,0.560976,1.000000
4,2025-04-01 08:00:00.000 UTC,0.496296,0.626866,2.029412,1.417910,0.569948,1.000000
...,...,...,...,...,...,...,...
4363,2026-03-30 14:00:00.000 UTC,0.457447,0.790698,1.911765,1.302326,0.471698,0.382353
4364,2026-03-30 16:00:00.000 UTC,0.393365,0.746988,1.851562,1.361446,0.500000,0.390625
4365,2026-03-30 18:00:00.000 UTC,0.411111,0.972973,1.735849,1.405405,0.490196,0.396226
4366,2026-03-30 20:00:00.000 UTC,0.401316,0.934426,1.472527,1.229508,0.486486,0.340659


In [12]:
temp_4 = adv.statistical_validation(DF_common_final, columns = user_cols, save=False)
display(temp_4[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrower_participation_rate,0.0,0.0,0.429398,0.152804,0.532396
1,repayment_discipline,0.0,0.0,0.740296,0.394095,1.203841
2,supplier_activity,0.0,0.0,1.887723,0.188323,2.514067
3,borrower_activity,0.0,0.0,1.432692,0.196811,1.949872
4,collateral_usage_rate,0.0,0.0,0.542547,0.075929,0.613436
5,collateral_adoption_rate,0.0,0.0,0.736416,0.255558,1.018182


In [13]:
# values will be variably sparse , because flashloan tx count col hads 47.5% of null values
DF_common_final["avg_flashloan_size_usd"] = DF_common_final["flashloan_amount_value_usd"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)
DF_common_final["avg_flashloan_size_eth"] = DF_common_final["flashloan_amount_value_eth"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)

DF_common_final["flashloan_fee_rate"] = DF_common_final["flashloan_premium_value_usd"] / DF_common_final["flashloan_amount_value_usd"].replace(0, pd.NA)

DF_common_final["flashloan_usage_intensity"] = DF_common_final["flashloan_tx_count"] / (
    DF_common_final["borrow_tx_count"] + DF_common_final["supply_tx_count"]
).replace(0, pd.NA)

DF_common_final["flashloan_user_activity"] = DF_common_final["flashloan_tx_count"] / DF_common_final["unique_flashloan_initiators"].replace(0, pd.NA)

DF_common_final["variable_debt_flashloan_ratio"] = DF_common_final["variable_flashloan_tx_count"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)  # ⚠️ 47.5% nulls

DF_common_final["no_debt_flashloan_ratio"] = DF_common_final["no_open_debt_flashloan_tx_count"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)

flashloand_cols = [
    "avg_flashloan_size_usd", "avg_flashloan_size_eth",
    "flashloan_fee_rate",
    "flashloan_usage_intensity",
    "flashloan_user_activity",
    "variable_debt_flashloan_ratio",
    "no_debt_flashloan_ratio",
]

DF_common_final[["time_bucket"] + flashloand_cols]

,time_bucket,avg_flashloan_size_usd,avg_flashloan_size_eth,flashloan_fee_rate,flashloan_usage_intensity,flashloan_user_activity,variable_debt_flashloan_ratio,no_debt_flashloan_ratio
0,2025-04-01 00:00:00.000 UTC,36720.841001,20.099236,0.00026,0.161491,1.3,0.269231,0.730769
1,2025-04-01 02:00:00.000 UTC,284037.728669,154.745585,0.000465,0.132184,1.210526,0.173913,0.826087
2,2025-04-01 04:00:00.000 UTC,28944.448637,15.722497,0.000032,0.075000,1.285714,0.111111,0.888889
3,2025-04-01 06:00:00.000 UTC,66836.817851,36.049638,0.00013,0.088083,2.428571,0.647059,0.352941
4,2025-04-01 08:00:00.000 UTC,76216.654241,40.554512,0.000332,0.120172,1.647059,0.392857,0.607143
...,...,...,...,...,...,...,...,...
4363,2026-03-30 14:00:00.000 UTC,9390.528687,4.550998,0.000499,0.068404,1.909091,0.095238,0.904762
4364,2026-03-30 16:00:00.000 UTC,324543.317928,157.697499,0.0005,0.031429,2.2,0.0,1.0
4365,2026-03-30 18:00:00.000 UTC,125787.03684,62.009188,0.000498,0.069444,2.0,0.1,0.9
4366,2026-03-30 20:00:00.000 UTC,299008.447133,147.339229,0.0005,0.043062,1.8,0.0,1.0


In [14]:
temp_5 = adv.statistical_validation(DF_common_final, columns = flashloand_cols, save=False)
display(temp_5[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,avg_flashloan_size_usd,0.1603,0.0000,119942.928498,5.168172,351823.684872
1,avg_flashloan_size_eth,0.1603,0.0000,39.562088,4.498859,110.842695
2,flashloan_fee_rate,0.1603,0.8255,0.000294,0.599335,0.000500
3,flashloan_usage_intensity,0.0000,0.1603,0.070436,0.543030,0.140713
4,flashloan_user_activity,0.1603,0.0000,1.877292,0.553890,3.833333
5,variable_debt_flashloan_ratio,0.1603,29.5804,0.251651,0.880069,0.625000
6,no_debt_flashloan_ratio,0.1603,0.2752,0.748349,0.295946,1.000000


In [15]:
tx_cols = [
    "supply_tx_count", "withdrawal_tx_count", "borrow_tx_count",
    "repay_tx_count", "flashloan_tx_count", "liquidation_tx_count"
]

# DF_common_final["total_activity"] = DF_common_final[tx_cols].fillna(0).sum(axis=1)

user_cols = ["unique_suppliers", "unique_borrowers", "unique_flashloan_initiators"]
DF_common_final["user_activity"] = DF_common_final[user_cols].fillna(0).sum(axis=1)

turnover_usd_cols = ["supply_amount_value_usd", "withdrawal_amount_value_usd", "borrow_amount_value_usd", "repay_amount_value_usd"]
turnover_eth_cols = ["supply_amount_value_eth", "withdrawal_amount_value_eth", "borrow_amount_value_eth", "repay_amount_value_eth"]
DF_common_final["protocol_turnover_usd"] = DF_common_final[turnover_usd_cols].fillna(0).sum(axis=1)
DF_common_final["protocol_turnover_eth"] = DF_common_final[turnover_eth_cols].fillna(0).sum(axis=1)

DF_common_final["leverage_indicator"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["supply_amount_value_usd"].replace(0, pd.NA)

# DF_common_final["market_stress_index_usd"] = (
#     DF_common_final["liquidation_tx_count"] *
#     DF_common_final["liquidation_debt_covered_value_usd"]
# ) / DF_common_final["borrow_amount_value_usd"].replace(0, pd.NA)  # ⚠️ 47% nulls
# DF_common_final["market_stress_index_eth"] = (
#     DF_common_final["liquidation_tx_count"] *
#     DF_common_final["liquidation_debt_covered_value_eth"]
# ) / DF_common_final["borrow_amount_value_eth"].replace(0, pd.NA)  # ⚠️ 47% nulls

# DF_common_final["capital_efficiency"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)  # ⚠️ 21% nulls

market_cols = [
    # "total_activity",
    "user_activity",
    "protocol_turnover_usd", "protocol_turnover_eth",
    "leverage_indicator",
    # "market_stress_index_usd", "market_stress_index_eth",
    # "capital_efficiency",
]

DF_common_final[["time_bucket"] + market_cols]

,time_bucket,user_activity,protocol_turnover_usd,protocol_turnover_eth,leverage_indicator
0,2025-04-01 00:00:00.000 UTC,111.0,3.058051e+24,1.673831e+21,1.056768e+16
1,2025-04-01 02:00:00.000 UTC,126.0,9.672094e+24,5.269417e+21,2.291481e+16
2,2025-04-01 04:00:00.000 UTC,82.0,1.116883e+25,6.066856e+21,1.890458e+16
3,2025-04-01 06:00:00.000 UTC,108.0,2.137041e+25,1.152648e+22,2.029154e+15
4,2025-04-01 08:00:00.000 UTC,152.0,1.146487e+25,6.100410e+21,2.087778e+16
...,...,...,...,...,...
4363,2026-03-30 14:00:00.000 UTC,199.0,2.281366e+25,1.105629e+22,2.222101e+17
4364,2026-03-30 16:00:00.000 UTC,216.0,1.092956e+25,5.310674e+21,4.733560e+16
4365,2026-03-30 18:00:00.000 UTC,190.0,7.226959e+24,3.562678e+21,1.915061e+17
4366,2026-03-30 20:00:00.000 UTC,157.0,5.469152e+25,2.694977e+22,2.560355e+17


In [16]:
temp_6 = adv.statistical_validation(DF_common_final, columns = market_cols, save=False)
display(temp_6[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,user_activity,0.0,0.0,1.826296e+02,0.426591,3.170000e+02
1,protocol_turnover_usd,0.0,0.0,3.019618e+25,2.599261,1.185907e+26
2,protocol_turnover_eth,0.0,0.0,1.023838e+22,2.572781,3.820324e+22
3,leverage_indicator,0.0,0.0,1.111108e+17,1.623511,4.336223e+17


In [17]:
print(f" {DF_common_1.shape[0]} rows x {DF_common_1.shape[1]} cols")
display(DF_common_1.head(PREVIEW_ROWS))

 54198 rows x 9 cols


,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045000,0.000000,0.045000,1.000000,1.123769,5
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6
5,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.046031,0.029176,0.045980,1.125029,1.164874,91
6,2025-04-01 00:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,0.089417,0.017815,0.088239,1.023239,1.127670,3
7,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.026735,0.020251,0.026735,1.042603,1.067234,96
8,2025-04-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,0.048260,0.032141,0.048260,1.119375,1.163539,73
9,2025-04-01 00:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,NaN,0.035663,0.054515,1.144648,1.189138,1


In [18]:
DF_common_1.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,object
asset,object
asset_symbol,object
last_borrow_rate,float64
liquidity_rate,float64
variable_borrow_rate,float64
liquidity_index,float64
variable_borrow_index,float64
update_count_liquidity,int64


In [19]:
#Rate dynamics, many cols have null values, adjusting for that.
DF_common_1["borrow_supply_spread"] = DF_common_1["last_borrow_rate"] - DF_common_1["liquidity_rate"]  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["variable_supply_spread"] = DF_common_1["variable_borrow_rate"] - DF_common_1["liquidity_rate"]

DF_common_1["borrow_rate_premium"] = DF_common_1["last_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 11.6% nulls + 5.9% zeros guarded

DF_common_1["variable_rate_premium"] = DF_common_1["variable_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

# DF_common_1["rate_change"] = DF_common_1["last_borrow_rate"] - DF_common_1["last_borrow_rate"].shift(1)  # ⚠️ 11.6% nulls
DF_common_1["rate_change"] = (DF_common_1["last_borrow_rate"] - DF_common_1["last_borrow_rate"].shift(1)) / DF_common_1["last_borrow_rate"].replace(0, pd.NA)

# DF_common_1["liquidity_rate_change"] = DF_common_1["liquidity_rate"] - DF_common_1["liquidity_rate"].shift(1)
DF_common_1["liquidity_rate_change"] = (DF_common_1["liquidity_rate"] - DF_common_1["liquidity_rate"].shift(1)) / DF_common_1["liquidity_rate"].replace(0, pd.NA)

DF_common_1["variable_borrow_rate_change"] = DF_common_1["variable_borrow_rate"] - DF_common_1["variable_borrow_rate"].shift(1)

rate_dynamics_cols = [
    "borrow_supply_spread",
    "variable_supply_spread",
    "borrow_rate_premium",
    "variable_rate_premium",
    "rate_change",
    "liquidity_rate_change",
    "variable_borrow_rate_change",
]

DF_common_1[["time_bucket"] + rate_dynamics_cols]

,time_bucket,borrow_supply_spread,variable_supply_spread,borrow_rate_premium,variable_rate_premium,rate_change,liquidity_rate_change,variable_borrow_rate_change
0,2025-04-01 00:00:00.000 UTC,0.004843,0.004841,19.607808,19.602141,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,0.045000,0.045000,<NA>,<NA>,0.886601,<NA>,0.039899
2,2025-04-01 00:00:00.000 UTC,0.022185,0.022185,3.010077,3.010076,-0.354533,1.0,-0.011778
3,2025-04-01 00:00:00.000 UTC,0.021494,0.021494,1.808023,1.808023,0.309244,0.585093,0.014873
4,2025-04-01 00:00:00.000 UTC,0.003795,0.003795,4.23801,4.238025,-8.681822,-21.694209,-0.043127
...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0.009802,0.009801,1.394598,1.394564,NaN,0.579001,-0.001623
54194,2026-03-30 22:00:00.000 UTC,0.005737,0.005737,1.345744,1.345743,-0.551345,-0.497,-0.012311
54195,2026-03-30 22:00:00.000 UTC,0.014329,0.014329,2.809029,2.809029,-0.003637,-1.094935,-0.000081
54196,2026-03-30 22:00:00.000 UTC,0.003035,0.003035,181.34336,181.342655,-6.291501,-469.719637,-0.019198


In [20]:
temp_7 = adv.statistical_validation(DF_common_1, columns = rate_dynamics_cols, save=False)
display(temp_7[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrow_supply_spread,16.4157,0.0000,1.847150e-02,1.695964,0.055000
1,variable_supply_spread,0.0000,0.0018,1.752195e-02,0.942400,0.055000
2,borrow_rate_premium,23.1503,0.0000,3.671160e+07,124.580208,83.177876
3,variable_rate_premium,7.5556,0.0000,1.559854e+03,57.999772,94.106102
4,rate_change,30.4255,0.0000,-2.494077e+02,54.879207,0.963385
5,liquidity_rate_change,7.5575,0.0000,-1.505631e+09,137.444463,1.000000
6,variable_borrow_rate_change,0.0018,0.0000,4.736688e-07,85192.392449,0.054838


In [21]:
# growth metrics — per-asset shift(1) (DF_common_1 is asset-level, so t-1 is the same asset's prev 6h bucket)
# index cols are >= 1.0 (no zeros / nulls), so no denominator guard needed
DF_common_1["liquidity_index_growth"] = (
    DF_common_1["liquidity_index"] - DF_common_1.groupby("asset")["liquidity_index"].shift(1)
) / DF_common_1.groupby("asset")["liquidity_index"].shift(1)

DF_common_1["variable_borrow_index_growth"] = (
    DF_common_1["variable_borrow_index"] - DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)
) / DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)

# interest / debt accrual velocity = per-asset delta of the index
DF_common_1["interest_accrual_velocity"] = DF_common_1["liquidity_index"] - DF_common_1.groupby("asset")["liquidity_index"].shift(1)
DF_common_1["debt_accrual_velocity"] = DF_common_1["variable_borrow_index"] - DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)

growth_cols = [
    "liquidity_index_growth",
    "variable_borrow_index_growth",
    "interest_accrual_velocity",
    "debt_accrual_velocity",
]

DF_common_1[["time_bucket"] + growth_cols]

,time_bucket,liquidity_index_growth,variable_borrow_index_growth,interest_accrual_velocity,debt_accrual_velocity
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
2,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
3,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,5.398111e-06,7.696396e-06,6.290285e-06,9.411540e-06
54194,2026-03-30 22:00:00.000 UTC,3.778214e-06,5.082917e-06,4.012313e-06,5.563039e-06
54195,2026-03-30 22:00:00.000 UTC,1.748105e-06,4.910503e-06,1.772118e-06,5.065598e-06
54196,2026-03-30 22:00:00.000 UTC,3.483201e-09,6.316533e-07,3.483878e-09,6.341862e-07


In [22]:
temp_8 = adv.statistical_validation(DF_common_1, columns = growth_cols, save=False)
display(temp_8[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,liquidity_index_growth,0.072,7.5758,0.000008,20.703815,0.000017
1,variable_borrow_index_growth,0.072,0.0018,0.000020,12.333003,0.000045
2,interest_accrual_velocity,0.072,7.5758,0.000009,21.568366,0.000018
3,debt_accrual_velocity,0.072,0.0018,0.000023,12.876553,0.000051


In [23]:
# rate risk metrics — same-unit ratios (no usd/eth split)
# liquidity_rate 5.9% zeros guarded, last_borrow_rate 11.6% nulls
DF_common_1["borrow_pressure"] = DF_common_1["variable_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

DF_common_1["lending_attractiveness"] = DF_common_1["liquidity_rate"] / DF_common_1["last_borrow_rate"].replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["reserve_stress_score"] = (
    DF_common_1["variable_borrow_rate"] - DF_common_1["liquidity_rate"]
) / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

rate_risk_cols = [
    "borrow_pressure",
    "lending_attractiveness",
    "reserve_stress_score",
]

DF_common_1[["time_bucket"] + rate_risk_cols]

,time_bucket,borrow_pressure,lending_attractiveness,reserve_stress_score
0,2025-04-01 00:00:00.000 UTC,19.602141,0.051000,18.602141
1,2025-04-01 00:00:00.000 UTC,<NA>,0.000000,<NA>
2,2025-04-01 00:00:00.000 UTC,3.010076,0.332217,2.010076
3,2025-04-01 00:00:00.000 UTC,1.808023,0.553090,0.808023
4,2025-04-01 00:00:00.000 UTC,4.238025,0.235960,3.238025
...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,1.394564,0.717052,0.394564
54194,2026-03-30 22:00:00.000 UTC,1.345743,0.743083,0.345743
54195,2026-03-30 22:00:00.000 UTC,2.809029,0.355995,1.809029
54196,2026-03-30 22:00:00.000 UTC,181.342655,0.005514,180.342655


In [24]:
temp_9 = adv.statistical_validation(DF_common_1, columns = rate_risk_cols, save=False)
display(temp_9[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrow_pressure,7.5556,0.0000,1559.853947,57.999772,94.106102
1,lending_attractiveness,16.4157,8.0572,0.376185,0.803611,0.776771
2,reserve_stress_score,7.5556,0.0000,1558.853947,58.036979,93.106102


In [25]:
# activity / volatility metrics — rolling std per asset over 4 × 6h (24h) windows
# update_count is already a metric, so not re-created here
DF_common_1["rate_volatility"] = DF_common_1.groupby("asset")["last_borrow_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["liquidity_volatility"] = DF_common_1.groupby("asset")["liquidity_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())

DF_common_1["borrow_volatility"] = DF_common_1.groupby("asset")["variable_borrow_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())

volatility_cols = [
    "rate_volatility",
    "liquidity_volatility",
    "borrow_volatility",
]

DF_common_1[["time_bucket"] + volatility_cols]

,time_bucket,rate_volatility,liquidity_volatility,borrow_volatility
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
2,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
3,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,5.245031e-04,7.409617e-04,5.246930e-04
54194,2026-03-30 22:00:00.000 UTC,1.047773e-05,1.460308e-05,9.825413e-06
54195,2026-03-30 22:00:00.000 UTC,5.177098e-05,4.262686e-05,5.973130e-05
54196,2026-03-30 22:00:00.000 UTC,5.263478e-08,4.418101e-09,1.226323e-07


In [26]:
temp_10 = adv.statistical_validation(DF_common_1, columns = volatility_cols, save=False)
display(temp_10[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,rate_volatility,5.705,7.7506,0.003056,9.348940,0.003116
1,liquidity_volatility,0.072,7.5611,0.000976,8.351547,0.002005
2,borrow_volatility,0.072,7.4337,0.001248,8.646683,0.002402


In [27]:
# momentum (ML) features — per-asset rate(t) / rate(t-k); k in {1, 4} (4 × 6h = 24h), denominators guarded
DF_common_1["momentum"] = DF_common_1["last_borrow_rate"] / DF_common_1.groupby("asset")["last_borrow_rate"].shift(1).replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["borrow_momentum_24h"] = DF_common_1["last_borrow_rate"] / DF_common_1.groupby("asset")["last_borrow_rate"].shift(4).replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["liquidity_momentum_24h"] = DF_common_1["liquidity_rate"] / DF_common_1.groupby("asset")["liquidity_rate"].shift(4).replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

DF_common_1["index_momentum_24h"] = DF_common_1["liquidity_index"] / DF_common_1.groupby("asset")["liquidity_index"].shift(4)  # index >= 1.0, no zero-guard needed

momentum_cols = [
    "momentum",
    "borrow_momentum_24h",
    "liquidity_momentum_24h",
    "index_momentum_24h",
]

DF_common_1[["time_bucket"] + momentum_cols]

,time_bucket,momentum,borrow_momentum_24h,liquidity_momentum_24h,index_momentum_24h
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
2,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
3,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,1.028276,1.040468,1.082638,1.000021
54194,2026-03-30 22:00:00.000 UTC,0.999324,0.998184,0.996379,1.000015
54195,2026-03-30 22:00:00.000 UTC,0.995356,0.995465,0.991697,1.000009
54196,2026-03-30 22:00:00.000 UTC,1.000015,0.999959,0.999731,1.000000


In [28]:
temp_11 = adv.statistical_validation(DF_common_1, columns = momentum_cols, save=False)
display(temp_11[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,momentum,27.0803,0.0,1.040278e+00,1.557873,1.028077
1,borrow_momentum_24h,27.4863,0.0,1.073158e+00,5.058654,1.073163
2,liquidity_momentum_24h,7.8305,0.0,7.032880e+06,222.517242,1.181667
3,index_momentum_24h,0.2823,0.0,1.000031e+00,0.000367,1.000065


In [29]:
# cross-asset features — rank within each time_bucket + asset share of updates (asset_symbol present)
DF_common_1["rank_by_borrow_rate"] = DF_common_1.groupby("time_bucket")["last_borrow_rate"].rank()  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["rank_by_liquidity_rate"] = DF_common_1.groupby("time_bucket")["liquidity_rate"].rank()

DF_common_1["asset_share_of_updates"] = DF_common_1["update_count_liquidity"] / DF_common_1.groupby("time_bucket")["update_count_liquidity"].transform("sum").replace(0, pd.NA)

cross_asset_cols = [
    "rank_by_borrow_rate",
    "rank_by_liquidity_rate",
    "asset_share_of_updates",
]

DF_common_1[["time_bucket"] + cross_asset_cols]

,time_bucket,rank_by_borrow_rate,rank_by_liquidity_rate,asset_share_of_updates
0,2025-04-01 00:00:00.000 UTC,2.0,2.0,0.069307
1,2025-04-01 00:00:00.000 UTC,5.0,1.0,0.016502
2,2025-04-01 00:00:00.000 UTC,4.0,4.0,0.016502
3,2025-04-01 00:00:00.000 UTC,7.0,7.0,0.006601
4,2025-04-01 00:00:00.000 UTC,1.0,3.0,0.019802
...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,8.0,12.0,0.356250
54194,2026-03-30 22:00:00.000 UTC,5.0,9.0,0.225000
54195,2026-03-30 22:00:00.000 UTC,4.0,4.0,0.006250
54196,2026-03-30 22:00:00.000 UTC,1.0,1.0,0.018750


In [30]:
temp_12 = adv.statistical_validation(DF_common_1, columns = cross_asset_cols, save=False)
display(temp_12[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,rank_by_borrow_rate,16.4157,0.0,5.913313,0.566461,12.000000
1,rank_by_liquidity_rate,0.0000,0.0,6.926510,0.568201,14.000000
2,asset_share_of_updates,0.0000,0.0,0.080593,1.363972,0.306846


In [31]:
DF_common_1.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,object
asset,object
asset_symbol,object
last_borrow_rate,float64
liquidity_rate,float64
variable_borrow_rate,float64
liquidity_index,float64
variable_borrow_index,float64
update_count_liquidity,int64
borrow_supply_spread,float64


In [32]:
# ── Separate sparse-family feature frames (liquidation + user-account risk) ───
# Liquidation & user-account series are too sparse for the 2h panel, so the
# liquidation/risk/market-stress metrics were left commented above. Here they are
# built on the DENSE 24h / 7d rollups instead, kept SEPARATE from the DF_common_* frames.
# Borrow-side denominators are aggregated from DF_common_final to the SAME grain.
import transform as tf
from data_validation import canonicalize_keys

borrow_ctx_cols = ["borrow_tx_count", "unique_borrowers",
                   "borrow_amount_value_usd", "borrow_amount_value_eth"]

def borrow_context(freq):
    ctx = tf.aggregate_by_time_bucket(DF_common_final, "time_bucket", borrow_ctx_cols,
                                      agg_func="sum", freq=freq)
    return canonicalize_keys(ctx)

def add_liquidation_features(liq, ctx):
    # liquidation-metric block (was commented) on the dense frame + grain-matched borrow context
    l = liq.merge(ctx, on="time_bucket", how="left")
    l["liquidation_rate"]             = l["liquidation_tx_count"] / l["borrow_tx_count"].replace(0, pd.NA)
    l["liquidation_volume_ratio_eth"] = l["liquidation_debt_covered_value_eth"] / l["borrow_amount_value_eth"].replace(0, pd.NA)
    l["liquidation_severity_usd"]     = l["liquidated_collateral_value_usd"] / l["liquidation_tx_count"].replace(0, pd.NA)
    l["liquidation_severity_eth"]     = l["liquidated_collateral_value_eth"] / l["liquidation_tx_count"].replace(0, pd.NA)
    l["avg_liquidation_debt_usd"]     = l["liquidation_debt_covered_value_usd"] / l["liquidation_tx_count"].replace(0, pd.NA)
    l["avg_liquidation_debt_eth"]     = l["liquidation_debt_covered_value_eth"] / l["liquidation_tx_count"].replace(0, pd.NA)
    l["liquidator_concentration"]     = l["unique_liquidators"] / l["liquidation_tx_count"].replace(0, pd.NA)
    l["liquidation_user_ratio"]       = l["unique_liquidated_users"] / l["unique_borrowers"].replace(0, pd.NA)
    l["market_stress_index_usd"]      = (l["liquidation_tx_count"] * l["liquidation_debt_covered_value_usd"]) / l["borrow_amount_value_usd"].replace(0, pd.NA)
    l["market_stress_index_eth"]      = (l["liquidation_tx_count"] * l["liquidation_debt_covered_value_eth"]) / l["borrow_amount_value_eth"].replace(0, pd.NA)
    return l

def add_risk_features(user, ctx):
    # risk-metric block (was commented) + debt_expansion + capital_efficiency, on user state
    u = user.merge(ctx[["time_bucket", "borrow_amount_value_usd"]], on="time_bucket", how="left")
    u["collateralization_ratio"]     = u["avg_total_collateral_base"] / u["avg_total_debt_base"].replace(0, pd.NA)
    u["borrow_capacity_utilization"] = u["avg_total_debt_base"] / (u["avg_total_debt_base"] + u["avg_available_borrows_base"]).replace(0, pd.NA)
    u["remaining_borrow_capacity"]   = u["avg_available_borrows_base"] / u["avg_total_collateral_base"].replace(0, pd.NA)
    u["risk_buffer"]                 = u["avg_current_liquidation_threshold"] - u["avg_ltv"]
    u["ltv_utilization"]             = u["avg_ltv"] / u["avg_current_liquidation_threshold"].replace(0, pd.NA)
    u["distance_to_liquidation"]     = 1 - u["ltv_utilization"]
    u["debt_expansion_ratio"]        = u["avg_total_debt_base"] / u["avg_total_collateral_base"].replace(0, pd.NA)
    u["capital_efficiency"]          = u["borrow_amount_value_usd"] / u["avg_total_collateral_base"].replace(0, pd.NA)
    # ⚠️ Flag.md repair: collateralization_ratio ~11% null (zero-debt sampled users);
    # debt_expansion_ratio already carries the inverted (debt/collateral) form, so only
    # the always-defined margin + sampled-borrow indicator are added here.
    u["overcollateral_margin"]       = u["avg_total_collateral_base"] - u["avg_total_debt_base"]
    u["has_sampled_borrow_position"] = (u["avg_total_debt_base"] > 0).astype(int)
    return u

DF_liq_features, DF_user_features = {}, {}
for g, liqf, userf in [("24h", df_liq_24h, df_user_account_24h), ("7d", df_liq_7d, df_user_account_7d)]:
    ctx = borrow_context(g)
    DF_liq_features[g]  = add_liquidation_features(liqf, ctx)
    DF_user_features[g] = add_risk_features(userf, ctx)
    print(f"[{g}] liq_features {DF_liq_features[g].shape} | user_features {DF_user_features[g].shape}")

[24h] liq_features (364, 24) | user_features (364, 21)
[7d] liq_features (52, 24) | user_features (52, 19)


In [33]:
liq_feature_cols = [
    "liquidation_rate", "liquidation_volume_ratio_eth",
    "liquidation_severity_usd", "liquidation_severity_eth",
    "avg_liquidation_debt_usd", "avg_liquidation_debt_eth",
    "liquidator_concentration", "liquidation_user_ratio",
    "market_stress_index_usd", "market_stress_index_eth",
]
risk_feature_cols = [
    "collateralization_ratio", "borrow_capacity_utilization", "remaining_borrow_capacity",
    "risk_buffer", "ltv_utilization", "distance_to_liquidation",
    "debt_expansion_ratio", "capital_efficiency",
    "overcollateral_margin", "has_sampled_borrow_position",
]

print("── liquidation features (24h) — dense vs the 72%-empty 2h panel ──")
display(adv.statistical_validation(DF_liq_features["24h"], columns=liq_feature_cols, save=False)[["column", "null_pct", "zero_pct", "negative_pct"]])
print("── risk features (24h) ──")
display(adv.statistical_validation(DF_user_features["24h"], columns=risk_feature_cols, save=False)[["column", "null_pct", "zero_pct", "negative_pct"]])

── liquidation features (24h) — dense vs the 72%-empty 2h panel ──


,column,null_pct,zero_pct,negative_pct
0,liquidation_rate,0.0000,8.5165,0.0
1,liquidation_volume_ratio_eth,0.0000,8.5165,0.0
2,liquidation_severity_usd,8.5165,0.0000,0.0
3,liquidation_severity_eth,8.5165,0.0000,0.0
4,avg_liquidation_debt_usd,8.5165,0.0000,0.0
5,avg_liquidation_debt_eth,8.5165,0.0000,0.0
6,liquidator_concentration,8.5165,0.0000,0.0
7,liquidation_user_ratio,0.0000,8.5165,0.0
8,market_stress_index_usd,0.0000,8.5165,0.0
9,market_stress_index_eth,0.0000,8.5165,0.0


── risk features (24h) ──


,column,null_pct,zero_pct,negative_pct
0,collateralization_ratio,1.0989,0.0000,0.0
1,borrow_capacity_utilization,0.0000,1.0989,0.0
2,remaining_borrow_capacity,0.0000,0.0000,0.0
3,risk_buffer,0.0000,0.0000,0.0
4,ltv_utilization,0.0000,0.0000,0.0
5,distance_to_liquidation,0.0000,0.0000,0.0
6,debt_expansion_ratio,0.0000,1.0989,0.0
7,capital_efficiency,0.0000,0.0000,0.0
8,overcollateral_margin,0.0000,0.0000,0.0
9,has_sampled_borrow_position,0.0000,1.0989,0.0


In [34]:
from pathlib import Path

OUT_DIR = Path("transformed_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# naming note (ERR-14): DF_common_2.csv = the RATE-FEATURE-ENRICHED asset panel
# (DF_common_1 + spreads/growth/volatility/momentum cols added above) — NOT the old
# reserve_config state panel the pre-redesign docs described (config dropped, ERR-13).
DF_common_2 = DF_common_1
DF_common_final_1 = DF_common_final

DF_common_final_1.to_csv(OUT_DIR / "DF_common_final_1.csv", index=False)
DF_common_2.to_csv(OUT_DIR / "DF_common_2.csv", index=False)

# separate sparse-family feature frames (kept apart from the DF_common_* frames)
DF_liq_features["24h"].to_csv(OUT_DIR / "DF_liq_features_24h.csv", index=False)
DF_liq_features["7d"].to_csv(OUT_DIR / "DF_liq_features_7d.csv", index=False)
DF_user_features["24h"].to_csv(OUT_DIR / "DF_user_features_24h.csv", index=False)
DF_user_features["7d"].to_csv(OUT_DIR / "DF_user_features_7d.csv", index=False)

print(f"wrote {len(DF_common_final_1)} + {len(DF_common_2)} rows to {OUT_DIR}/")
print("wrote 4 separate feature frames (liq/user x 24h/7d) to", OUT_DIR)

wrote 4368 + 54198 rows to transformed_data/
wrote 4 separate feature frames (liq/user x 24h/7d) to transformed_data
